# AI University Knowledge Hub — Full Capstone Demo

This notebook runs the entire project end to end, printing a **short, plain-language result** after every stage — instead of the raw framework logs Kafka, Spark, Airflow, and Great Expectations normally produce.

**Training program:** Modern Data Engineering for AI Systems — SDAIA Academy

**The five stages:**
1. Document ingestion and validation (Kafka + Pydantic)
2. Governed storage Bronze ← Silver ← Gold (Delta Lake)
3. Intelligent question-answering (RAG)
4. Full automatic orchestration (Apache Airflow)
5. Final quality gate and audit trail (Great Expectations + OpenLineage)

> Note: running this notebook end to end takes about 10-15 minutes (mostly downloading AI models and starting Spark).

In [1]:
import os

if not os.path.exists('/content/ai-university-knowledge-hub'):
    !git clone -q https://github.com/LujainAldujain/University_Hub.git /content/ai-university-knowledge-hub

%cd /content/ai-university-knowledge-hub
print('Project downloaded successfully')

/content/ai-university-knowledge-hub
Project downloaded successfully


In [2]:
AIRFLOW_VERSION = '3.3.0'
import sys
PYTHON_VERSION = f'{sys.version_info.major}.{sys.version_info.minor}'
CONSTRAINT_URL = f'https://raw.githubusercontent.com/apache/airflow/constraints-{AIRFLOW_VERSION}/constraints-{PYTHON_VERSION}.txt'

!pip install -q apache-airflow=={AIRFLOW_VERSION} --constraint {CONSTRAINT_URL}
!pip install -q -r requirements.txt
print('✅ All required packages installed')

✅ All required packages installed


In [3]:
import logging

for noisy in ['kafka', 'py4j', 'great_expectations', 'openlineage', 'chromadb',
              'sentence_transformers', 'httpx', 'urllib3', 'airflow']:
    logging.getLogger(noisy).setLevel(logging.ERROR)

from loguru import logger as _loguru_logger
_loguru_logger.remove()

def section(title):
    print()
    print('=' * 60)
    print('  ' + title)
    print('=' * 60)

def ok(msg):
    print('✅ ' + msg)

def bad(msg):
    print('❌ ' + msg)

def info(msg):
    print('ℹ️  ' + msg)

print('Clean-output helpers ready')

Clean-output helpers ready


In [5]:
!rm -rf /tmp/kraft-combined-logs

In [6]:
section('Infrastructure setup: starting a local Kafka broker')

!curl -sSOL https://downloads.apache.org/kafka/4.2.1/kafka_2.13-4.2.1.tgz
!tar -xzf kafka_2.13-4.2.1.tgz

import subprocess
import time

kafka_dir = 'kafka_2.13-4.2.1'
cluster_id = subprocess.check_output([kafka_dir + '/bin/kafka-storage.sh', 'random-uuid']).decode().strip()
subprocess.run([kafka_dir + '/bin/kafka-storage.sh', 'format', '-t', cluster_id,
                '-c', kafka_dir + '/config/server.properties', '--standalone'], check=True)
subprocess.Popen([kafka_dir + '/bin/kafka-server-start.sh', kafka_dir + '/config/server.properties'],
                  stdout=open('/content/kafka.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(12)

for topic in ['university.documents.raw', 'university.documents.validated', 'university.documents.dlq']:
    subprocess.run([kafka_dir + '/bin/kafka-topics.sh', '--create', '--topic', topic,
                    '--bootstrap-server', 'localhost:9092', '--partitions', '3', '--replication-factor', '1'],
                   check=True, capture_output=True)

ok('Kafka broker is running, with all 3 topics created (raw / validated / dead-letter)')


  Infrastructure setup: starting a local Kafka broker
✅ Kafka broker is running, with all 3 topics created (raw / validated / dead-letter)


In [7]:
import os

os.environ['AIRFLOW_HOME'] = '/content/ai-university-knowledge-hub/airflow'
os.environ['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'

!airflow db migrate > /content/airflow_setup.log 2>&1
!python scripts/seed_sample_documents.py > /content/seed.log 2>&1

ok('Airflow metadata DB initialized and sample documents seeded')

✅ Airflow metadata DB initialized and sample documents seeded


## Stage 1: Document ingestion and validation
The system receives documents and checks each one (file type, size, completeness) before accepting it.

In [8]:
import sys
sys.path.insert(0, '/content/ai-university-knowledge-hub')

from producer.document_producer import main as produce
from consumer.document_consumer import main as consume

section('Stage 1: Document ingestion and validation')

produce()
result = consume()

ok(str(result['total']) + ' documents sent')
ok(str(result['accepted']) + ' accepted')
bad(str(result['rejected']) + ' rejected')
print()
info('Rejection reasons:')
for reason, count in result['rejection_reasons'].items():
    print('   • ' + reason + '  (x' + str(count) + ')')


  Stage 1: Document ingestion and validation


/tmp/ipykernel_21216/231435289.py:9: DeprecationWarning: key_serializer does not implement kafka.serializer.Serializer
  produce()
/tmp/ipykernel_21216/231435289.py:9: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  produce()
/tmp/ipykernel_21216/231435289.py:10: DeprecationWarning: key_serializer does not implement kafka.serializer.Serializer
  result = consume()
/tmp/ipykernel_21216/231435289.py:10: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  result = consume()


✅ 10 documents sent
✅ 5 accepted
❌ 5 rejected

ℹ️  Rejection reasons:
   • Input should be 'pdf', 'docx' or 'txt'  (x1)
   • Value error, file_size_bytes must be > 0  (x1)
   • Value error, declared file_type 'pdf' does not match the actual extension '.txt'  (x1)
   • Value error, file_path '/content/ai-university-knowledge-hub/incoming_documents/does_not_exist_on_disk.txt' does not exist on disk  (x1)
   • Field required  (x1)


## Stage 2: Governed storage (Bronze ← Silver ← Gold)
Accepted documents are cleaned and stored through three layers: **raw (Bronze)** ← **governed (Silver)** ← **summarized (Gold)**, using Delta Lake.

In [9]:
from lakehouse.bronze.bronze_loader import main as bronze_main
from lakehouse.silver.silver_transform import main as silver_main
from lakehouse.gold.gold_aggregate import main as gold_main

section('Stage 2: Governed storage (Bronze ← Silver ← Gold)')

b = bronze_main()
ok(str(b['row_count']) + ' documents saved to the raw layer (Bronze)')

s = silver_main()
if s.get('revision_applied_this_run'):
    ok('A previously-seen document was updated in place (not duplicated)')
if s.get('schema_enforcement_confirmed'):
    ok('The system automatically rejected a file that tried to add an unknown data column (data integrity protection)')
ok('The governed layer (Silver) now holds ' + str(s['row_count']) + ' clean documents')

g = gold_main()
ok('Summarized ' + str(g['silver_row_count']) + ' documents into ' + str(g['gold_row_count']) + ' knowledge categories ready for analysis')


  Stage 2: Governed storage (Bronze ← Silver ← Gold)
+------------------------------------+------------------------------+---------+--------------+---------------+
|document_id                         |file_name                     |file_type|category      |file_size_bytes|
+------------------------------------+------------------------------+---------+--------------+---------------+
|626da374-fb5b-402d-b18b-13d181a912a3|attendance_policy.pdf         |pdf      |policy        |2354           |
|c98e1a48-8499-4516-943c-2addb4df831f|cs_program_requirements.docx  |docx     |course_catalog|37357          |
|3262ae9d-b515-4206-ac1e-09fc909acd95|graduation_requirements.txt   |txt      |policy        |1956           |
|d8af9c1e-49b7-4ac3-ae64-242c1bddce8a|course_withdrawal_policy.txt  |txt      |policy        |1502           |
|ecd90c8f-eb0d-4aaf-9029-1be17d12e122|scholarships_financial_aid.txt|txt      |scholarship   |1864           |
|cb525693-8024-45e6-a9d8-c7b2910cbb08|cs_program_requireme

In [10]:
!pip install -U torch torchvision transformers sentence-transformers accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 20.5 MB/s eta 0:00:00


## Stage 3: Intelligent question-answering (RAG)
The system answers student questions using **only** the official documents on file — it never makes up information.

In [11]:
from rag.rag_pipeline import RAGPipeline, EXAMPLE_QUESTIONS

section('Stage 3: Intelligent question-answering (RAG)')
info('Building the smart index (may take a minute to download the AI models)...')

pipeline = RAGPipeline()
ok('The system is ready to answer questions')
print()

for q in EXAMPLE_QUESTIONS:
    result = pipeline.answer(q)
    print('❓ Question: ' + q)
    print('💬 Answer: ' + result['answer'])
    if result['citations']:
        print('📄 Source: ' + ', '.join(result['citations']))
    else:
        print('📄 Source: none — the system admitted it does not know instead of making up an answer')
    print('-' * 60)


  Stage 3: Intelligent question-answering (RAG)
ℹ️  Building the smart index (may take a minute to download the AI models)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

❓ Question: What are the graduation requirements?
💬 Answer: Based on the provided university graduation requirements, students must complete a minimum of 132 total credit hours. Additionally, the final 30 credit hours of a student's program must be completed at this university, and a senior capstone project or comprehensive exam is required in the final year for most majors. [Source 1]
📄 Source: graduation_requirements.txt
------------------------------------------------------------
❓ Question: What courses are required for Computer Science?
💬 Answer: According to the provided document, the required courses for the Bachelor of Science in Computer Science include:

1. **Introduction to Programming** (CS 101) - 3 credit hours
2. **Data Structures and Algorithms** (CS 102) - 3 credit hours
3. **Discrete Mathematics** (CS 201) - 3 credit hours
4. **Computer Organization and Architecture** (CS 210) - 3 credit hours
5. **Object-Oriented Software Design** (CS 220) - 3 credit hours
6. **Databa

## Stage 5: Final quality gate and audit trail
Before any data is trusted, the system runs **6 strict quality checks** (Great Expectations). It also records a start/success/failure event for every stage (OpenLineage) — a complete audit trail of every operation.

In [15]:
from quality.ge_checkpoint import run_silver_quality_checkpoint
from lineage.lineage_emitter import emit_lineage

section('Stage 5: Final quality gate (Great Expectations)')

with emit_lineage('quality_checkpoint_demo'):
    outcome = run_silver_quality_checkpoint()

for r in outcome['results']:
    label = '✅' if r['success'] else '❌'
    print(label + ' ' + r['expectation'] + '  (column: ' + str(r['column']) + ')')

print()
if outcome['success']:
    ok('All 6 quality checks passed — the data is safe to use')
else:
    bad('A quality check failed — processing must stop before the next stage')

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp6oa6f76c' for ephemeral docs site



  Stage 5: Final quality gate (Great Expectations)


Calculating Metrics:   0%|          | 0/42 [00:00<?, ?it/s]

  return column.astype(str).str.contains(regex)



✅ expect_column_values_to_not_be_null  (column: document_id)
✅ expect_column_values_to_not_be_null  (column: file_name)
✅ expect_column_values_to_be_unique  (column: file_name)
✅ expect_column_values_to_match_regex  (column: file_type)
✅ expect_column_values_to_be_between  (column: word_count)
✅ expect_column_values_to_be_in_set  (column: is_valid)

✅ All 6 quality checks passed — the data is safe to use


### Test: what happens if the data is broken?
To prove the quality gate is **real** and not just for show, we deliberately inject a genuine data error (a duplicate document) and confirm the system halts processing automatically.

In [13]:
from deltalake import DeltaTable, write_deltalake
import pyarrow as pa

section('Test: protecting the system from broken data')

SILVER_PATH = '/content/ai-university-knowledge-hub/lakehouse/silver/documents_silver'
dt = DeltaTable(SILVER_PATH)
tbl = dt.to_pyarrow_table()
row = tbl.to_pylist()[0]
row['document_id'] = 'test-duplicate-0000'
write_deltalake(SILVER_PATH, pa.Table.from_pylist([row], schema=tbl.schema), mode='append')
info('Deliberately injected a duplicate document to simulate a real bug (file: ' + row['file_name'] + ')')

gold_ran = False
try:
    with emit_lineage('quality_checkpoint_failure_demo'):
        result = run_silver_quality_checkpoint()
        if not result['success']:
            failed = [r['expectation'] for r in result['results'] if not r['success']]
            raise RuntimeError('Check failed: ' + str(failed))
    gold_ran = True
    gold_main()
except RuntimeError:
    bad('Processing halted automatically — the broken data will not be promoted')

print()
if not gold_ran:
    ok('The system successfully protected itself from broken data')

dt = DeltaTable(SILVER_PATH)
dt.delete(predicate="document_id = 'test-duplicate-0000'")
ok('Data restored to its correct state')


  Test: protecting the system from broken data


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp3ia9ktym' for ephemeral docs site


ℹ️  Deliberately injected a duplicate document to simulate a real bug (file: attendance_policy.pdf)


Calculating Metrics:   0%|          | 0/42 [00:00<?, ?it/s]

  return column.astype(str).str.contains(regex)



❌ Processing halted automatically — the broken data will not be promoted

✅ The system successfully protected itself from broken data
✅ Data restored to its correct state


## Stage 4: Full automatic orchestration (Apache Airflow)
Everything above was run manually, step by step, for a clear walkthrough. Now let's run **the exact same project fully automatically**, through a real scheduler used in production at large companies (Apache Airflow) — one command, same dependency order, so any downstream stage automatically stops if an earlier one fails.

In [14]:
import subprocess
from datetime import date

section('Stage 4: Running the full pipeline automatically via Airflow')
info('Running now (may take a couple of minutes)...')

run_date = date.today().isoformat()
proc = subprocess.run(
    ['airflow', 'dags', 'test', 'university_knowledge_hub_pipeline', run_date],
    capture_output=True, text=True,
)
raw_output = proc.stdout + proc.stderr
with open('/content/airflow_dag_test_raw.log', 'w', encoding='utf-8') as f:
    f.write(raw_output)

TASK_LABELS = {
    'produce_documents': 'Send documents',
    'validate_and_consume': 'Validate and classify',
    'check_ingestion_quality': 'Batch acceptance gate',
    'bronze_load': 'Raw storage (Bronze)',
    'silver_transform': 'Clean and update (Silver)',
    'quality_checkpoint': 'Quality check (Great Expectations)',
    'gold_aggregate': 'Knowledge summary (Gold)',
}

task_names = []
returns = []
for line in raw_output.splitlines():
    if 'Current task name:' in line:
        task_names.append(line.split('Current task name:', 1)[1].strip())
    if 'Returned value was:' in line:
        returns.append(line.split('Returned value was:', 1)[1].strip())

run_success = 'state=success' in raw_output

print()
if task_names:
    for name in task_names:
        label = TASK_LABELS.get(name, name)
        ok(label)
else:
    info('Could not find task details in the log — check the full file')

print()
if run_success:
    ok('The entire project completed automatically via Airflow!')
else:
    bad('A stage failed — check the full log: /content/airflow_dag_test_raw.log')

info('The full technical log is saved at: /content/airflow_dag_test_raw.log (for anyone who wants the technical details)')


  Stage 4: Running the full pipeline automatically via Airflow
ℹ️  Running now (may take a couple of minutes)...

ℹ️  Could not find task details in the log — check the full file

✅ The entire project completed automatically via Airflow!
ℹ️  The full technical log is saved at: /content/airflow_dag_test_raw.log (for anyone who wants the technical details)


## ✅ Summary
The **AI University Knowledge Hub** ran successfully end to end — from receiving documents to answering student questions, through every quality and audit gate — first manually step by step, then fully automatically via Airflow.

**The most important point:** the system answers accurately when the information exists in the official documents, and admits it doesn't know instead of making up an answer when it doesn't.

Capstone project — **Modern Data Engineering for AI Systems**, SDAIA Academy
Full code: https://github.com/LujainAldujain/University_Hub